# DisgraPhi Bootstrap Training on Google Colab\n\nTrain a general handwriting recognition adapter on the IAM dataset.\n\n**Hardware Requirements:**\n- GPU: T4 (16GB) - Free tier\n- RAM: 12GB\n- Disk: ~15GB\n\n**Estimated Time:**\n- Setup: 5-10 minutes\n- Training: 3-5 hours (3 epochs on IAM dataset)\n\n**What you'll get:**\n- Trained LoRA adapter (~50-200MB)\n- Uploaded to HuggingFace for easy sharing\n- Ready for personalization

## 1. Setup Environment\n\nInstall dependencies and check GPU availability.

In [ ]:
# Check GPU\n!nvidia-smi\n\nimport torch\nprint(f"CUDA available: {torch.cuda.is_available()}")\nif torch.cuda.is_available():\n    print(f"GPU: {torch.cuda.get_device_name(0)}")\n    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")\nelse:\n    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

In [ ]:
# Mount Google Drive for persistent storage\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\n# Create project directory\nimport os\nPROJECT_DIR = '/content/drive/MyDrive/disgraphi'\nos.makedirs(PROJECT_DIR, exist_ok=True)\nos.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)\nos.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)\nprint(f"✓ Project directory: {PROJECT_DIR}")

In [ ]:
# Clone DisgraPhi repository\n!git clone https://github.com/velocitatem/disgraPhi.git /content/disgraphi\n%cd /content/disgraphi

In [ ]:
# Install dependencies\nprint("Installing dependencies (this may take 3-5 minutes)...")\n!pip install -q -r ml/requirements.txt\nprint("✓ Dependencies installed")

## 2. Configure Training\n\nSet up model and training parameters.

In [ ]:
# Training configuration\nCONFIG = {\n    # Model\n    'model_provider': 'deepseek-ocr',  # Options: deepseek-ocr, smolvlm-256m, qwen3-vl-2b\n    'lora_r': 16,\n    'lora_alpha': 32,\n    'load_in_4bit': True,\n    \n    # Dataset\n    'dataset_type': 'iam',\n    'sample_ratio': 1.0,  # Use full dataset (set to 0.1 for quick test)\n    \n    # Training\n    'num_train_epochs': 3,\n    'per_device_train_batch_size': 3,  # T4 can handle batch=3\n    'gradient_accumulation_steps': 4,\n    'learning_rate': 2e-4,\n    \n    # Memory optimizations\n    'use_liger_kernel': True,\n    'optim': 'adamw_8bit',\n    \n    # Output\n    'output_dir': f'{PROJECT_DIR}/checkpoints',\n    'data_dir': f'{PROJECT_DIR}/data/iam',\n}\n\nprint("Configuration:")\nfor key, value in CONFIG.items():\n    print(f"  {key}: {value}")

## 3. Download IAM Dataset\n\nDownload and cache the IAM handwriting dataset (~5GB).

In [ ]:
# IAM dataset will auto-download on first run\n# This cell checks if it's already cached in Drive\n\nimport os\nfrom pathlib import Path\n\niam_cache = Path(CONFIG['data_dir'])\nif (iam_cache / 'processed').exists():\n    print("✓ IAM dataset found in Drive cache")\n    print(f"  Location: {iam_cache}")\nelse:\n    print("IAM dataset not cached. It will download on first training run.")\n    print("This is a one-time download (~5GB, 10-15 minutes)")

## 4. Authenticate with HuggingFace\n\nRequired to upload your trained adapter.

In [ ]:
from huggingface_hub import notebook_login\n\nprint("Authenticate with HuggingFace to upload your adapter.")\nprint("Get your token at: https://huggingface.co/settings/tokens")\n\nnotebook_login()

## 5. Train Model\n\nStart bootstrap training (3-5 hours on T4).

In [ ]:
# Build training command\ncmd = f"""python ml/models/train_trl.py \\\n  --model_provider {CONFIG['model_provider']} \\\n  --lora_r {CONFIG['lora_r']} \\\n  --lora_alpha {CONFIG['lora_alpha']} \\\n  {'--load_in_4bit' if CONFIG['load_in_4bit'] else ''} \\\n  --dataset_type {CONFIG['dataset_type']} \\\n  --data_dir {CONFIG['data_dir']} \\\n  --sample_ratio {CONFIG['sample_ratio']} \\\n  --num_train_epochs {CONFIG['num_train_epochs']} \\\n  --per_device_train_batch_size {CONFIG['per_device_train_batch_size']} \\\n  --gradient_accumulation_steps {CONFIG['gradient_accumulation_steps']} \\\n  --learning_rate {CONFIG['learning_rate']} \\\n  --output_dir {CONFIG['output_dir']} \\\n  --optim {CONFIG['optim']} \\\n  {'--use_liger_kernel' if CONFIG.get('use_liger_kernel') else ''}\n"""\n\nprint("Starting training...")\nprint("This will take 3-5 hours. You can monitor progress below.")\nprint("="*80)\n\n!{cmd}

## 6. Evaluate Results\n\nCheck training metrics and test the model.

In [ ]:
# Find latest experiment\nimport os\nfrom pathlib import Path\n\ncheckpoint_dir = Path(CONFIG['output_dir'])\nexperiments = sorted([d for d in checkpoint_dir.iterdir() if d.is_dir()], key=lambda x: x.stat().st_mtime)\n\nif experiments:\n    latest_exp = experiments[-1]\n    print(f"Latest experiment: {latest_exp.name}")\n    \n    adapter_path = latest_exp / 'final_adapter'\n    if adapter_path.exists():\n        print(f"✓ Adapter saved: {adapter_path}")\n        \n        # Check adapter size\n        size_mb = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file()) / 1024**2\n        print(f"  Size: {size_mb:.1f} MB")\n    else:\n        print("⚠️ Adapter not found")\nelse:\n    print("No experiments found")

In [ ]:
# Load TensorBoard logs\n%load_ext tensorboard\nlog_dir = checkpoint_dir / 'logs'\n%tensorboard --logdir {log_dir}

## 7. Test Inference\n\nTry the trained model on a sample image.

In [ ]:
# Load trained model\nfrom ml.models.providers import create_model\n\nprint("Loading trained model...")\nmodel = create_model(\n    model_type=CONFIG['model_provider'],\n    load_in_4bit=True,\n    bootstrap_adapter_path=str(adapter_path)\n)\nprint("✓ Model loaded")

In [ ]:
# Test on sample image\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\n# Get a test sample from IAM\nfrom ml.data.datasets import IAMDataset\ntest_dataset = IAMDataset(data_dir=CONFIG['data_dir'], split='test')\n\n# Get random sample\nimport random\nidx = random.randint(0, len(test_dataset) - 1)\nsample = test_dataset[idx]\n\n# Display image\nplt.figure(figsize=(12, 3))\nplt.imshow(sample['image'], cmap='gray')\nplt.axis('off')\nplt.title('Test Image')\nplt.show()\n\n# Generate prediction\nprint("\nGenerating transcription...")\nprediction = model.generate(sample['image'], prompt="Transcribe this handwritten text:")\n\nprint(f"\nGround truth: {sample['text']}")\nprint(f"Prediction:   {prediction}")\n\n# Calculate CER\nfrom ml.models.eval import compute_ocr_metrics\nmetrics = compute_ocr_metrics(sample['text'], prediction)\nprint(f"\nCER: {metrics['cer']:.3f}")\nprint(f"WER: {metrics['wer']:.3f}")

## 8. Upload to HuggingFace\n\nShare your trained adapter.

In [ ]:
from huggingface_hub import HfApi, create_repo\n\n# Configuration\nrepo_name = f"disgraphi-bootstrap-{latest_exp.name}"\nusername = "YOUR_USERNAME"  # Replace with your HuggingFace username\nrepo_id = f"{username}/{repo_name}"\n\nprint(f"Uploading to: https://huggingface.co/{repo_id}")\n\n# Create repository\ntry:\n    create_repo(repo_id, exist_ok=True, private=False)\n    print("✓ Repository created")\nexcept Exception as e:\n    print(f"Warning: {e}")\n\n# Upload adapter\napi = HfApi()\napi.upload_folder(\n    folder_path=str(adapter_path),\n    repo_id=repo_id,\n    commit_message="Upload bootstrap adapter"\n)\n\nprint(f"\n✓ Upload complete!")\nprint(f"  View at: https://huggingface.co/{repo_id}")

## 9. Download Adapter (Optional)\n\nDownload the adapter to use locally.

In [ ]:
# Create ZIP archive\nimport shutil\n\nzip_path = f'/content/{latest_exp.name}_adapter'\nshutil.make_archive(zip_path, 'zip', adapter_path)\n\nprint(f"✓ Adapter archived: {zip_path}.zip")\nprint(f"  Size: {Path(zip_path + '.zip').stat().st_size / 1024**2:.1f} MB")\nprint("\nDownload from the Files panel (left sidebar)")

## Next Steps\n\n1. **Personalization**: Use the `colab_personalization.ipynb` notebook to fine-tune on your handwriting\n2. **Local deployment**: Download the adapter and use it with the DisgraPhi inference server\n3. **Experiment**: Try different models (SmolVLM, Qwen3-VL) or hyperparameters\n\n**Need help?** Check the [DisgraPhi documentation](https://github.com/velocitatem/disgraPhi)